<img src="https://raw.githubusercontent.com/Databricks-BR/lab_agosto_2025/main/images/head_lab.png">

## LAB 03 — Entregável Operações (challenge 002)

Diagnóstico + proposta de automação + protótipo (este dashboard). Dataset 2 (`Topic_group`) **ausente** — não inventar.

1. Rode o **LAB 01** antes.
2. Widgets iguais.
3. **Run All**. Horas = `Time to Resolution` − `First Response Time`, só Closed.
4. Fixe os `display` no AI/BI. Não exponha `Customer Email`.

In [ ]:
def widget_or_default(name: str, default: str) -> str:
    try:
        dbutils.widgets.text(name, default)
        value = dbutils.widgets.get(name)
        return value if value else default
    except NameError:
        return default


catalog_name = widget_or_default("catalog_name", "workspace")
schema_name = widget_or_default("schema_name", "supportiq")
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")
print(f"{catalog_name}.{schema_name}")

### Tile 1 — Onde o fluxo trava (p95 horas, Closed)

In [ ]:
display(
    spark.sql(
        """
SELECT
  `Ticket Channel`,
  `Ticket Priority`,
  `Ticket Type`,
  COUNT(*) AS closed_tickets,
  ROUND(AVG((unix_timestamp(`Time to Resolution`) - unix_timestamp(`First Response Time`)) / 3600.0), 2) AS avg_hours,
  ROUND(PERCENTILE_APPROX((unix_timestamp(`Time to Resolution`) - unix_timestamp(`First Response Time`)) / 3600.0, 0.95), 2) AS p95_hours,
  ROUND(AVG(`Customer Satisfaction Rating`), 2) AS avg_csat
FROM customer_support_tickets
WHERE `Ticket Status` = 'Closed'
  AND `Time to Resolution` IS NOT NULL
  AND `First Response Time` IS NOT NULL
GROUP BY `Ticket Channel`, `Ticket Priority`, `Ticket Type`
ORDER BY p95_hours DESC
LIMIT 25
"""
    )
)

### Tile 2 — O que move CSAT (só Closed)

In [ ]:
display(
    spark.sql(
        """
SELECT
  `Ticket Channel`,
  `Ticket Type`,
  `Ticket Priority`,
  COUNT(*) AS closed,
  ROUND(AVG(`Customer Satisfaction Rating`), 2) AS avg_csat,
  ROUND(AVG((unix_timestamp(`Time to Resolution`) - unix_timestamp(`First Response Time`)) / 3600.0), 2) AS avg_hours
FROM customer_support_tickets
WHERE `Ticket Status` = 'Closed'
GROUP BY `Ticket Channel`, `Ticket Type`, `Ticket Priority`
ORDER BY avg_csat ASC, closed DESC
"""
    )
)

### Tile 3 — Horas desperdiçadas (soma Closed) + fila aberta

In [ ]:
display(
    spark.sql(
        """
SELECT
  'closed_hours' AS slice,
  COUNT(*) AS tickets,
  ROUND(SUM((unix_timestamp(`Time to Resolution`) - unix_timestamp(`First Response Time`)) / 3600.0), 1) AS hours,
  SUM(CASE WHEN `Ticket Priority` IN ('Critical', 'High') THEN 1 ELSE 0 END) AS high_or_critical
FROM customer_support_tickets
WHERE `Ticket Status` = 'Closed'
  AND `Time to Resolution` IS NOT NULL
  AND `First Response Time` IS NOT NULL
UNION ALL
SELECT
  'open_or_pending' AS slice,
  COUNT(*) AS tickets,
  NULL AS hours,
  SUM(CASE WHEN `Ticket Priority` IN ('Critical', 'High') THEN 1 ELSE 0 END) AS high_or_critical
FROM customer_support_tickets
WHERE `Ticket Status` IN ('Open', 'Pending Customer Response')
"""
    )
)

### Protótipo — roteador (tipo × prioridade × texto)

Não é classificador de 8 `Topic_group` (Dataset 2 ausente). Roteia com colunas reais + comprimento do texto.

In [ ]:
display(
    spark.sql(
        """
SELECT
  `Ticket ID`,
  `Ticket Type`,
  `Ticket Priority`,
  `Ticket Channel`,
  `Ticket Status`,
  CASE
    WHEN `Ticket Priority` = 'Critical' THEN 'HUMAN_NOW'
    WHEN `Ticket Type` IN ('Cancellation request', 'Refund request')
         AND length(`Ticket Description`) > 400 THEN 'HUMAN_REVIEW'
    WHEN `Ticket Type` IN ('Billing inquiry', 'Product inquiry')
         AND `Ticket Priority` IN ('Low', 'Medium') THEN 'AI_SUGGEST'
    ELSE 'HUMAN_QUEUE'
  END AS route,
  length(`Ticket Description`) AS description_len
FROM customer_support_tickets
WHERE `Ticket Status` IN ('Open', 'Pending Customer Response')
ORDER BY
  CASE `Ticket Priority` WHEN 'Critical' THEN 1 WHEN 'High' THEN 2 WHEN 'Medium' THEN 3 ELSE 4 END,
  description_len DESC
LIMIT 40
"""
    )
)

### Tile 4 — Ranking da rota na fila aberta

Conta `HUMAN_NOW`, `HUMAN_REVIEW`, `AI_SUGGEST` e `HUMAN_QUEUE` só em Open / Pending. Critical continua humano.

In [ ]:
display(
    spark.sql(
        """
SELECT route, COUNT(*) AS tickets
FROM (
  SELECT
    CASE
      WHEN `Ticket Priority` = 'Critical' THEN 'HUMAN_NOW'
      WHEN `Ticket Type` IN ('Cancellation request', 'Refund request')
           AND length(`Ticket Description`) > 400 THEN 'HUMAN_REVIEW'
      WHEN `Ticket Type` IN ('Billing inquiry', 'Product inquiry')
           AND `Ticket Priority` IN ('Low', 'Medium') THEN 'AI_SUGGEST'
      ELSE 'HUMAN_QUEUE'
    END AS route
  FROM customer_support_tickets
  WHERE `Ticket Status` IN ('Open', 'Pending Customer Response')
)
GROUP BY route
ORDER BY tickets DESC
"""
    )
)

### Automação (o que sim / não)

**Automatizar:** triagem `AI_SUGGEST` (Billing / Product inquiry, Low–Medium); rascunho de resposta a partir de `Resolution` de Closed similares no mesmo `Ticket Type` (Lab 04).

**Não automatizar:** `Critical`; `Cancellation request` / `Refund request` com descrição longa — julgamento e dinheiro. Sem Dataset 2 não há ground truth de 8 classes.

**Fluxo:** ticket entra → regra `route` → IA só em `AI_SUGGEST` → humano em `HUMAN_*` → fecha e mede horas + CSAT (Closed).

Custo: o dump **não tem** salário. Estime horas do tile 3 × custo/hora que o Diretor informar.

### Painéis gráficos

Ranking, galeria, classificação, segmentação, segmentação de imagem, estatísticas do modelo, matriz de confusão, curva de aprendizado e viés/variância. Calculados só com colunas reais do dump.


In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / '03_LAB_Dashboard', HERE.parent / 'lib', HERE.parent.parent / 'lib'):
    if (candidate / 'lab03_panels' / '__init__.py').exists():
        sys.path.insert(0, str(candidate))
        break
else:
    raise FileNotFoundError('lab03_panels não encontrado')

from lab03_panels import build_support_panels
from lab03_panels.render import render_panels

try:
    show = display
except NameError:
    show = print

panels = build_support_panels()
render_panels(panels, show)
print('secoes', list(panels.panels))
